# Customer Churn Prediction with a Decision Tree
**Assignment mini project | Decision Tree vs Logistic Regression**

> The included dataset is synthetic and for learning only. Do not report its results as real-world business performance.

## Learning objectives
1. Load and inspect a dataset.
2. Handle missing values without data leakage.
3. Split into training and testing sets.
4. Train a Decision Tree Classifier and Logistic Regression.
5. Evaluate with accuracy, precision, recall, F1-score and confusion matrices.
6. Interpret feature importance and the decision-tree plot.

## 1. Import libraries and load the data
We use pandas for tables, matplotlib for plots, and scikit-learn for machine learning.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, confusion_matrix, classification_report
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "customer_churn_synthetic.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (1200, 8)


,age,purchase_frequency,total_spending,days_since_last_purchase,support_interactions,membership_months,discount_usage_rate,churn
0,22.0,9,505.60,74,1.0,66,0.231,0
1,59.0,8,318.20,40,1.0,69,0.489,0
2,52.0,6,267.19,116,1.0,65,0.580,0
3,41.0,6,214.97,46,1.0,56,0.147,0
4,40.0,2,126.45,46,2.0,36,0.329,0


## 2. Understand and inspect the data
`churn = 1` means churned; `churn = 0` means retained. Missing predictor values are handled later by a median imputer fitted only on training data.

In [6]:
print("Column data types:")
display(df.dtypes.to_frame("data_type"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Target distribution:")
display(df["churn"].value_counts().sort_index().rename(index={0:"Retained (0)", 1:"Churned (1)"}).to_frame("count"))

Column data types:


,data_type
age,float64
purchase_frequency,int64
total_spending,float64
days_since_last_purchase,int64
support_interactions,float64
membership_months,int64
discount_usage_rate,float64
churn,int64


Missing values:


,missing_count
age,12
purchase_frequency,0
total_spending,12
days_since_last_purchase,0
support_interactions,12
membership_months,0
discount_usage_rate,0
churn,0


Target distribution:


,count
churn,
Retained (0),984
Churned (1),216


## 3. Separate features and target, then split
`X` contains predictor columns; `y` contains the target. Stratification keeps the churn proportion similar in train and test sets.

In [11]:
TARGET = "churn"
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df.drop(columns=TARGET)
y = df[TARGET]
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (960, 7)
Testing shape: (240, 7)


## 4. Build the model pipelines

**Why a pipeline?** The median imputer (and scaler for Logistic Regression) learns its parameters from the training data only. This prevents test-set information from leaking into training.

- Decision Trees do not generally need feature scaling.
- Logistic Regression benefits from standardization.
- `max_depth`, `min_samples_split`, and `min_samples_leaf` constrain tree complexity.

In [12]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

import numpy as np
import pandas as pd
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

print("All imports successful.")

Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
Python executable: f:\ML\AI_Engineering\AI-Engineering-Bootcamp\ai_eng_env\python.exe
NumPy: 2.5.1
Pandas: 3.0.3
Scikit-learn: 1.9.0
All imports successful.


In [13]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nFeature columns:")
print(X_train.columns.tolist())

print("\nData types:")
print(X_train.dtypes)

print("\nMissing values:")
print(X_train.isna().sum())

print("\nTarget classes:")
print(y_train.value_counts())

X_train shape: (960, 7)
y_train shape: (960,)

Feature columns:
['age', 'purchase_frequency', 'total_spending', 'days_since_last_purchase', 'support_interactions', 'membership_months', 'discount_usage_rate']

Data types:
age                         float64
purchase_frequency            int64
total_spending              float64
days_since_last_purchase      int64
support_interactions        float64
membership_months             int64
discount_usage_rate         float64
dtype: object

Missing values:
age                         11
purchase_frequency           0
total_spending              12
days_since_last_purchase     0
support_interactions        11
membership_months            0
discount_usage_rate          0
dtype: int64

Target classes:
churn
0    787
1    173
Name: count, dtype: int64


In [14]:
tree_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    )
])

decision_tree = Pipeline([
    ("preprocess", tree_preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

print("Pipeline created.")

decision_tree.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Pipeline created.
Decision Tree trained successfully.


In [15]:
logistic_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    )
])

logistic_regression = Pipeline([
    ("preprocess", logistic_preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Pipeline created.")

logistic_regression.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

: 

In [5]:
tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_features)
])

logistic_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)
])

decision_tree = Pipeline([
    ("preprocess", tree_preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_regression = Pipeline([
    ("preprocess", logistic_preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

models = {
    "Decision Tree": decision_tree,
    "Logistic Regression": logistic_regression
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained.")

: 

In [ ]:
## Why the kernel crashed

That message is generic: it means the Python process stopped unexpectedly. It does **not** identify the cause. From the cells shown, there is no clear code error that should crash the kernel; ordinary problems such as a missing file or column usually produce a traceback without killing it.

Possible causes include:

- The CSV is unexpectedly large, causing memory exhaustion during `read_csv`, `display(df)`, or model fitting.
- The Jupyter process ran out of available memory.
- An incompatible or damaged Python package installation.
- A lower-level issue reported in the Jupyter log.

The tree plot is limited to depth 4, so it is not an obvious cause in this notebook.

## How to find and fix it

1. Restart the kernel, then run cells one at a time to identify which cell triggers the crash.
2. Check the dataset size before displaying or training:

```python
print("File size (MB):", DATA_PATH.stat().st_size / 1_000_000)
print("Data shape:", df.shape)
```

If the file is very large, load a sample first:

```python
df = pd.read_csv(DATA_PATH, nrows=10_000)
```

3. If a particular cell consistently crashes, check the Jupyter log and system memory at that point. The log is needed to identify the **specific** cause.
4. If it happens immediately, restart Jupyter and verify that `numpy`, `pandas`, `matplotlib`, and `scikit-learn` are installed in the same environment as the notebook.

## 5. Evaluation metrics

- **Accuracy:** fraction of all predictions that are correct.
- **Precision for churn:** among customers predicted as churned, how many actually churned?
- **Recall for churn:** among customers who churned, how many did the model identify?
- **F1-score:** harmonic mean of precision and recall.

For customer churn, recall can be especially important because a false negative is a churned customer the system failed to flag.

In [ ]:
metric_rows = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    metric_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (churn=1)": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall (churn=1)": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1-score (churn=1)": f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    })
    print("\n", name)
    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Retained (0)", "Churned (1)"],
        zero_division=0
    ))

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

## 6. Plot confusion matrices
Rows show actual classes and columns show predicted classes. The matrix is ordered as retained (0), churned (1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        labels=[0, 1],
        display_labels=["Retained (0)", "Churned (1)"],
        values_format="d",
        colorbar=False,
        ax=ax
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 7. Feature importance

The Decision Tree's impurity-based feature importance measures how much each feature contributed to reducing impurity across the fitted tree. It is not causal proof and may be misleading when predictors are correlated.

In [ ]:
fitted_tree = decision_tree.named_steps["model"]
importance = pd.Series(
    fitted_tree.feature_importances_,
    index=numeric_features
).sort_values()

display(importance.sort_values(ascending=False).rename("importance").to_frame())

importance.plot(kind="barh", figsize=(8, 5), title="Decision Tree Feature Importance")
plt.xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()

## 8. Visualize and interpret the decision tree

At each decision node, the tree checks a condition such as `days_since_last_purchase <= threshold`.
- Follow the left branch when the condition is true.
- Follow the right branch when the condition is false.
- A leaf gives the predicted class.
- `gini` measures node impurity; `samples` is the number of training rows at the node; `value` shows class counts.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    fitted_tree,
    feature_names=numeric_features,
    class_names=["Retained (0)", "Churned (1)"],
    filled=True,
    rounded=True,
    impurity=True,
    fontsize=8,
    ax=ax
)
plt.title("Customer Churn Decision Tree (max_depth=4)")
plt.tight_layout()
plt.show()

## 9. Compare models and discuss limitations

Use the metrics table and confusion matrices to compare the models. Do not automatically select the model with the highest accuracy: churn costs, false negatives, false positives, and class balance matter.

**Decision Tree advantages:** visual and interpretable rules, non-linear relationships, feature interactions, no scaling requirement.

**Decision Tree limitations:** can overfit, can be sensitive to small data changes, and impurity-based importance is not causal.

**Logistic Regression:** useful linear baseline, benefits from scaling, and is often interpreted through coefficients; it may not capture nonlinear patterns unless features are engineered.

## 10. Assignment conclusion
This notebook demonstrates preprocessing, model implementation, evaluation, feature importance, tree visualization, and a Logistic Regression comparison. The included data is synthetic, so validate on a real, authorized dataset before making business conclusions.